In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [2]:
import re
from pyspark.sql import SparkSession

INPUT_PATH = r"G:\\My Drive\\Documents\\NLP\\data\\c4-train.00000-of-01024-30K.json\\c4-train.00000-of-01024-30K.json"
TOKEN_PATTERN = re.compile(r"[^\W_]+(?:['-][^\W_]+)*", re.UNICODE)

spark = (
    SparkSession.builder
    .appName("C4Vocabulary")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

In [3]:
documents = spark.read.json(INPUT_PATH).select("text").rdd

tokens = (
    documents
    .filter(lambda row: row.text is not None)
    .flatMap(lambda row: TOKEN_PATTERN.findall(row.text.lower()))
    .distinct()
)

document_count = documents.count()
vocabulary_size = tokens.count()

print(f"Documents: {document_count:,}")
print(f"Vocabulary size: {vocabulary_size:,}")

Documents: 30,000
Vocabulary size: 234,081


In [4]:
spark.stop()